# Survey modelling — portfolio baseline
Open with an R kernel. Run from this folder; install randomForest and jsonlite.
A new held-out baseline; results are separate from the original coursework. Not a diagnostic or causal model.


In [1]:
library(randomForest)
library(jsonlite)
# Fit preprocessing on training data and apply the same rules to validation.
prepare_features <- function(a,b) {
  for (name in names(a)) {
    if (is.numeric(a[[name]])) {
      m <- median(a[[name]],na.rm=TRUE); if(!is.finite(m))m<-0
      a[[name]][!is.finite(a[[name]])]<-m;b[[name]][!is.finite(b[[name]])]<-m
    } else {
      x<-as.character(a[[name]]);y<-as.character(b[[name]])
      x[is.na(x)]<-"__MISSING__";y[is.na(y)]<-"__MISSING__"
      lev<-unique(c(sort(unique(x)),"__OTHER__"));y[!y%in%lev]<-"__OTHER__"
      a[[name]]<-factor(x,levels=lev);b[[name]]<-factor(y,levels=lev)
    }
  }
  keep<-vapply(a,function(x)length(unique(x))>1&&(!is.factor(x)||nlevels(x)<=53),logical(1))
  list(train=a[,keep,drop=FALSE],valid=b[,keep,drop=FALSE])
}
run_task<-function(file,target,classification=FALSE) {
  d<-read.csv(file,check.names=TRUE);d<-d[!is.na(d[[target]]),]
  set.seed(42);idx<-sample(seq_len(nrow(d)),floor(.8*nrow(d)))
  cols<-setdiff(names(d),c(target,"RowIndex","X","dataset"))
  p<-prepare_features(d[idx,cols,drop=FALSE],d[-idx,cols,drop=FALSE]);y<-d[[target]][idx];truth<-d[[target]][-idx]
  if(classification)y<-factor(y)
  set.seed(42);model<-randomForest(x=p$train,y=y,ntree=200,importance=TRUE);pred<-predict(model,p$valid)
  if(classification) {
    labels<-sort(unique(c(as.character(truth),as.character(pred))))
    score<-mean(vapply(labels,function(k){tp<-sum(truth==k&pred==k);fp<-sum(truth!=k&pred==k);fn<-sum(truth==k&pred!=k);if(2*tp+fp+fn==0)0 else 2*tp/(2*tp+fp+fn)},numeric(1)))
    majority<-names(sort(table(y),decreasing=TRUE))[1]
    return(list(train_rows=length(idx),validation_rows=length(truth),accuracy=mean(pred==truth),macro_f1=score,majority_accuracy=mean(truth==majority)))
  }
  list(train_rows=length(idx),validation_rows=length(truth),rmse=sqrt(mean((pred-truth)^2)),mean_baseline_rmse=sqrt(mean((mean(y)-truth)^2)))
}
r<-list(kind="new Random Forest baseline",split="80/20 fixed holdout, seed 42; training-only preprocessing",regression=run_task("data/regression_train.csv","happiness"),classification=run_task("data/classification_train.csv","perfectMentalHealth",TRUE),limitations="Single random split; association, not diagnosis or causation.")
dir.create("outputs",showWarnings=FALSE);write_json(r,"outputs/survey-results.json",pretty=TRUE,auto_unbox=TRUE);print(r)


randomForest 4.7-1.2
Type rfNews() to see new features/changes/bug fixes.
$kind
[1] "new Random Forest baseline"

$split
[1] "80/20 fixed holdout, seed 42; training-only preprocessing"

$regression
$regression$train_rows
[1] 400

$regression$validation_rows
[1] 100

$regression$rmse
[1] 7.88075

$regression$mean_baseline_rmse
[1] 12.78512


$classification
$classification$train_rows
[1] 400

$classification$validation_rows
[1] 100

$classification$accuracy
[1] 0.36

$classification$macro_f1
[1] 0.3132821

$classification$majority_accuracy
[1] 0.28


$limitations
[1] "Single random split; association, not diagnosis or causation."

